# 04: EDA and Feature Engineering, LGD Model

This notebook covers the exploratory data analysis and feature engineering for the LGD model features. All analysis is performed on the defaulted loans of the development sample (matured loans, see `01_data_diagnosis`). Outliers are treated and the cleaned dataframe is saved for training.

**Input:** `data/processed/checkpoint_DataFrame_for_EDA.pkl`, `data/processed/features_LGD_for_EDA.pkl`  
**Output:** `data/processed/checkpoint_df_LGD_training.pkl`, `data/processed/features_LGD_num_for_training.pkl`, `data/processed/features_LGD_cat_for_training.pkl`

## Libraries

In [ ]:
# Run from the repository root so that data/, models/ and src/ resolve the same way
# whether the notebook is opened from the root or from production_notebooks/.
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "production_notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os

from feature_plot import plot_boxplots_list, plot_categoricals
from feature_statistics import dataset_audit, outlier_summary
from clean_features_LGD import fit_treatment_LGD

## 1. Load Checkpoint

In [ ]:
with open("data/processed/checkpoint_DataFrame_for_EDA.pkl", "rb") as f:
    data = pickle.load(f)
df_master_features = data["df_master_features"]

with open("data/processed/features_LGD_for_EDA.pkl", "rb") as f:
    features_LGD = pickle.load(f)

# LGD model trains exclusively on defaulted loans of the development sample
df_LGD = df_master_features.loc[
    (df_master_features["Target_Variable_Default"] == 1) & (df_master_features["is_matured"] == 1),
    ["Target_Variable_Loss"] + features_LGD
].copy()

print(f"df_LGD shape: {df_LGD.shape}")
print(f"Features: {features_LGD}")

## 2. Initial Audit

In [ ]:
audit_LGD = dataset_audit(df_LGD[features_LGD])
audit_LGD

## 3. Numeric Features

### 3.1 Distribution Overview

In [ ]:
# Binary or categorical columns stored as numbers are one-hot encoded, not scaled
CATEGORICAL_NUMERIC = ["loan_term_months"]

num_features_LGD = [
    f for f in df_LGD[features_LGD].select_dtypes(include=[np.number]).columns
    if f not in CATEGORICAL_NUMERIC
]
cat_features_LGD = [f for f in features_LGD if f not in num_features_LGD]
print(f"Numeric: {num_features_LGD}")
print(f"Categorical: {cat_features_LGD}")

plot_boxplots_list(df_LGD, num_features_LGD, cols=4)

### 3.2 Outlier Summary

In [ ]:
outlier_summary_LGD = outlier_summary(df_LGD, num_features_LGD)
outlier_summary_LGD

### 3.3 Upper Tails

For every numeric feature: skewness, the share of values beyond the extreme fence (Q3 + 3 IQR), the gap between p99 and the maximum, and the skewness after log1p. These are the quantities behind the rules in `LGD_TREATMENTS`: a p99 cap where the tail is long and discontinuous (max far above p99), and no log1p where it would invert the skew (mass near zero).

In [ ]:
rows = []
for f in num_features_LGD:
    s = df_LGD[f].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    fence = q3 + 3 * (q3 - q1)
    rows.append({
        "feature": f,
        "skew": s.skew(),
        "skew_log1p": np.log1p(s.clip(lower=0)).skew() if (s >= 0).all() else np.nan,
        "extreme_out_pct": (s > fence).mean() * 100,
        "p99": s.quantile(0.99),
        "max": s.max(),
        "zero_pct": (s == 0).mean() * 100,
        "nan_pct": df_LGD[f].isna().mean() * 100,
    })
tails_LGD = pd.DataFrame(rows).set_index("feature").round(3)
tails_LGD

### 3.4 Apply Treatments

The treatment of every feature the selection can return is declared once, in `LGD_TREATMENTS` (`src/clean_features_LGD.py`), and `FeaturePipeline_LGD` reuses the same specification for portfolio scoring:

- balances with long upper tails: cap at p99, no log1p (it inverts the skew);
- income: cap at p99, then log1p if the skew is still above 2;
- utilisation ratios: economic ceiling at 100;
- counts with a discontinuous upper tail: cap at p99;
- debt-to-income: negatives set to 0; credit history start year: floor at p1;
- remaining numeric NaNs: median of the fit population.

The table below lists what was applied to the features selected in this run.

In [ ]:
treatment_LGD = fit_treatment_LGD(df_LGD[features_LGD])
df_LGD_clean = df_LGD[["Target_Variable_Loss"]].join(treatment_LGD.transform(df_LGD[features_LGD]))
treatment_LGD.summary()

In [ ]:
# Post-treatment audit
audit_LGD_num_clean = dataset_audit(df_LGD_clean[num_features_LGD])
audit_LGD_num_clean

In [ ]:
# Post-treatment boxplots
plot_boxplots_list(df_LGD_clean, num_features_LGD, cols=4)

In [ ]:
# Post-treatment outlier summary
outlier_summary_LGD_num_clean = outlier_summary(df_LGD_clean, num_features_LGD)
outlier_summary_LGD_num_clean

## 4. Categorical Features

Categorical features (text categories and `loan_term_months`) are one-hot encoded at training time. Their distribution on the defaulted development sample:

In [ ]:
for col in cat_features_LGD:
    print(df_LGD_clean[col].value_counts(normalize=True).round(4), "\n")
plot_categoricals(df_LGD_clean, cat_features_LGD)

In [ ]:
print(f"df_LGD_clean: {df_LGD_clean.shape}")

## 5. Checkpoint: Save LGD Training Dataframe

In [ ]:
os.makedirs("data/processed", exist_ok=True)

with open("data/processed/checkpoint_df_LGD_training.pkl", "wb") as f:
    pickle.dump({"df_LGD_clean": df_LGD_clean}, f)

with open("data/processed/features_LGD_num_for_training.pkl", "wb") as f:
    pickle.dump(num_features_LGD, f)

with open("data/processed/features_LGD_cat_for_training.pkl", "wb") as f:
    pickle.dump(cat_features_LGD, f)

print(f"Saved. Shape: {df_LGD_clean.shape}")

In [ ]:
# Load checkpoint (skip re-running)
with open("data/processed/checkpoint_df_LGD_training.pkl", "rb") as f:
    data = pickle.load(f)
df_LGD_clean = data["df_LGD_clean"]

with open("data/processed/features_LGD_num_for_training.pkl", "rb") as f:
    num_features_LGD = pickle.load(f)

with open("data/processed/features_LGD_cat_for_training.pkl", "rb") as f:
    cat_features_LGD = pickle.load(f)